In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/gayatrijoshi663@gmail.com/regis-healthcare/1_setup/utility

In [0]:
# %run /Workspace/Users/dhotepatil00@gmail.com/regis-healthcare/1_setup/utility

In [0]:
print(bronze_schema,silver_schema,gold_schema) 

In [0]:
dbutils.widgets.text("catalog","regis_healthcare","catalog")
dbutils.widgets.text("data_source","employees","data_source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

#### Silver Processing

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
display(df_bronze)
print(df_bronze.count())

In [0]:
# schema check
print(df_bronze.count())
df_bronze.printSchema()

In [0]:
df_bronze.columns

In [0]:
# drop duplicate
df_silver = df_bronze.dropDuplicates()
print(df_silver.count())

In [0]:
df_silver = df_silver.withColumn(
    "employee_id",
    F.trim(F.col("employee_id"))
).withColumn(
    "first_name",
    F.trim(F.col("first_name"))
).withColumn(
    "last_name",
    F.trim(F.col("last_name"))
).withColumn(
    "job_title",
    F.trim(F.col("job_title"))
).withColumn(
    "facility_id",
    F.trim(F.col("facility_id"))
).withColumn(
    "phone",
    F.trim(F.col("phone"))
).withColumn(
    "email",
    F.trim(F.col("email"))
).withColumn(
    "hire_date",
    F.trim(F.col("hire_date"))
).withColumn(
    "employment_type",
    F.trim(F.col("employment_type"))
).withColumn(
    "status",
    F.trim(F.col("status"))
).withColumn(
    "salary",
    F.trim(F.col("salary"))
).withColumn(
    "created_at",
    F.trim(F.col("created_at"))
)

In [0]:
# null records count 
from pyspark.sql.functions import col,count,when
null_count = df_silver.select([count(when(col(c).isNull(),c)).alias(c)for c in df_silver.columns
                               ])
display(null_count)

#### Cleaning data in table

In [0]:
# 'employee_id',

check = df_silver.filter(~col("employee_id").rlike("^EMP"))
display(check)

df_silver = df_silver.withColumn("employee_id",when(~col("employee_id").rlike("^EMP"),None).otherwise(col("employee_id")))
display(df_silver)


In [0]:
#  'first_name',

from pyspark.sql.functions import col,when,trim,initcap

df_silver = df_silver.withColumn("first_name",initcap(trim(col("first_name"))))

dup = df_silver.groupBy("first_name").count().filter(col("count")>1)
dup.display()




In [0]:
#  'last_name'
from pyspark.sql.functions import col,when,trim,initcap

df_silver = df_silver.withColumn("first_name",initcap(trim(col("first_name"))))

dup = df_silver.groupBy("first_name").count().filter(col("count")>1)
dup.display()

In [0]:
#  'job_title',

from pyspark.sql.functions import when, lit, col, lower, trim

invalid_values = ["null", "nan", "n/a", "#n/a", "none", ""]

df_silver = df_silver.withColumn("job_title",
    when(
        col("job_title").isNull() | lower(trim(col("job_title"))).isin(invalid_values),
        lit("Unknown")
    ).otherwise(trim(col("job_title")))
)

display(df_silver)

In [0]:
#  'facility_id',

from pyspark.sql.functions import col,when
df_filt = df_silver.filter(~col("facility_id").rlike("^FAC"))

df_silver = df_silver.withColumn(
    "facility_id",
    when(
        (col("facility_id").isNull()) | (~col("facility_id").rlike("^FAC")),
        "0"
    ).otherwise(col("facility_id"))
)

display(df_filt)
display(df_silver)

In [0]:
#  'phone',

from pyspark.sql.functions import regexp_replace, col
df_silver = df_silver.withColumn("phone",
    regexp_replace(col("phone"), "[^0-9]", "")
)
display(df_silver)
from pyspark.sql.functions import col

df_silver = df_silver.filter(col("phone").rlike("^[0-9]{10}$")
)
display(df_silver)


df_silver = df_silver.withColumn(
    "phone",
    when( col("phone").isNull()| lower(trim(col("phone"))).isin(invalid_values),
        lit("0")
    ).otherwise(trim(col("phone")))
)

display(df_silver)


In [0]:
#  'email',
from pyspark.sql.functions import col, when, lower, trim, lit

invalid_values = ["null", "nan", "n/a", "#n/a", "none", ""]

df_silver = df_silver.withColumn("email",when(
        col("email").isNull() |
        lower(trim(col("email"))).isin(invalid_values),
        lit("unknown@email.com")
    ).otherwise(lower(trim(col("email"))))
)
display(df_silver)

In [0]:
#  'hire_date',

dup= df_silver.groupBy("hire_date").count()
display(dup)

In [0]:
#  'employment_type',

from pyspark.sql.functions import col, when, lower, trim, lit

invalid_values = ["null", "nan", "n/a", "#n/a", "none", ""]

df_silver = df_silver.withColumn(
    "employment_type",
    when(col("employment_type").isNull() |
        lower(trim(col("employment_type"))).isin(invalid_values),lit("not provided")
    ).otherwise(lower(trim(col("employment_type"))))
)
# display(df_silver)
dup= df_silver.groupBy("employment_type").count()
# display(dup)

In [0]:
#  'status',
from pyspark.sql.functions import col, when, lower, trim, lit

invalid_values = ["null", "nan", "n/a", "#n/a", "none", ""]

df_silver = df_silver.withColumn("status",
    when(col("status").isNull() |
        lower(trim(col("status"))).isin(invalid_values),lit("not active")
    ).otherwise(lower(trim(col("status"))))
)
# display(df_silver)
dup= df_silver.groupBy("status").count()
# display(dup)

In [0]:
#salary
df_silver = df_silver.withColumn("salary", abs(col("salary")))

display(df_silver)

In [0]:
#  'created_at',
dup= df_silver.groupBy("created_at").count()
display(dup)

In [0]:
# 'employee_id',
#  'first_name',
#  'last_name',
#  'job_title',
#  'facility_id',
#  'phone',
#  'email',
#  'hire_date',
#  'employment_type',
#  'status',
#  'salary',
#  'created_at',

silver table load

In [0]:
df_silver.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed","true")\
            .option("mergeSchema","true")\
                .option("overwriteSchema","true")\
            .mode("overwrite")\
               .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

dt = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source};")
print(dt.count())
display(dt)

In [0]:
# load to s3
df_silver.write.format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
    .mode("overwrite")\
    .partitionBy("current_date")\
    .save(f"s3://regis-healthcare/silver-clean-data/{data_source}/")